In [3]:
nums = (1, 2, 3)
print(id(nums), type(nums))

139171556675200 <class 'tuple'>


In [4]:
nums += (4,)

In [5]:
print(id(nums), type(nums))

139171557008672 <class 'tuple'>


In [6]:
a = [1, 2]
print(id(a), type(a))

139171556932416 <class 'list'>


In [7]:
a.append(3)
print(id(a), type(a))

139171556932416 <class 'list'>


-   tuple este imutable, imi face un obiect nou

In [8]:
a = [1, 2, 3]
b = [3]
a & b

TypeError: unsupported operand type(s) for &: 'list' and 'list'

-   pe seturi nu pot face intersectie, reuniune, diferenta, diferenta simetrica

In [9]:
a = {1, 2, 3}
b = {3}
print(a & b) # inters
print(a | b) # union
print(a - b) # dif
print(a ^ b) # dif sim

{3}
{1, 2, 3}
{1, 2}
{1, 2}


In [11]:
a = [1, 2, 3]
patr = [a**2 for a in a]
patr

[1, 4, 9]

In [18]:
a = [1, 2, 3, 4, 5, 6]
b = [x**2 if x % 2 == 0 else x for x in a]
b

[1, 4, 3, 16, 5, 36]

In [19]:
litere_unice = {c for c in "mississipi"}
litere_unice

{'i', 'm', 'p', 's'}

In [20]:
l = {word: len(word) for word in ['da', 'caca']}
l

{'da': 2, 'caca': 4}

In [22]:
sq = lambda x: x**2
sq(5)

25

In [ ]:
nums = [1, 4, 9, 16, 25]
a = list(map(lambda x: int(x**(1/2)), nums))
b = list(filter(lambda x: x % 2 == 0, nums))
c = list(map(lambda x: int(x**(1/2)), filter(lambda x: x % 2 == 0, nums)))



[2, 4]

In [4]:
names = ["Ana", "Ion", "Maria"]
scores = [95, 87, 92]
combined = list(zip(names, scores))
combined

[('Ana', 95), ('Ion', 87), ('Maria', 92)]

In [5]:
d = dict(combined)
d

{'Ana': 95, 'Ion': 87, 'Maria': 92}

In [6]:
d = dict(zip(names, scores))
d

{'Ana': 95, 'Ion': 87, 'Maria': 92}

In [7]:
s = sorted(d, key=lambda x: len(x), reverse=True)
s

['Maria', 'Ana', 'Ion']

In [8]:
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b
        
gen = fibonacci()
first_10 = [next(gen) for _ in range(10)]
first_10

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

In [9]:
import sys

In [10]:
big_sq_gen = (x**2 for x in range(100))
sys.getsizeof(big_sq_gen)

200

In [11]:
lst = [x**2 for x in range(100)]
sys.getsizeof(lst)

920

In [13]:
list(big_sq_gen)[0]

0

In [16]:
sys.getsizeof(list(big_sq_gen))

56

-   Perfect pentru stream processing

-   Lazy evaluation din Spark e inspirat din acelasi principiu ca generatoarele din Python -> nu calculezi nimic pana nu ai nevoie de rezultat

In [17]:
class DataPipeline:
    # Class variable — partajat între toate instanțele
    pipeline_count = 0

    def __init__(self, name: str, source: str):
        self.name = name           # public
        self.source = source       # public
        self._records = []         # protected (convenție)
        self.__id = id(self)       # private (name mangling)
        DataPipeline.pipeline_count += 1

    # Regular method
    def load(self, data: list):
        self._records = data
        return self                # method chaining

    def filter_records(self, condition):
        self._records = [r for r in self._records if condition(r)]
        return self

    # Property — getter
    @property
    def record_count(self):
        return len(self._records)

    # Setter
    @record_count.setter
    def record_count(self, value):
        raise AttributeError("Cannot set record_count directly")

    # Class method — operează pe clasă, nu pe instanță
    @classmethod
    def get_pipeline_count(cls):
        return cls.pipeline_count

    # Static method — nu are acces la instanță sau clasă
    @staticmethod
    def validate_source(source: str) -> bool:
        return source.startswith(("s3://", "hdfs://", "gs://"))

    # Dunder methods
    def __len__(self):
        return len(self._records)

    def __repr__(self):
        return f"DataPipeline(name='{self.name}', records={len(self)})"

    def __str__(self):
        return f"Pipeline '{self.name}' cu {len(self)} înregistrări"

    def __contains__(self, item):
        return item in self._records

    def __iter__(self):
        return iter(self._records)


# Moștenire
class BatchPipeline(DataPipeline):
    def __init__(self, name, source, batch_size=1000):
        super().__init__(name, source)  # apelează __init__ al părintelui
        self.batch_size = batch_size

    def process_in_batches(self):
        for i in range(0, len(self._records), self.batch_size):
            batch = self._records[i:i + self.batch_size]
            yield batch   # generator!

# Utilizare
p = DataPipeline("ETL_Job", "s3://bucket/data")
p.load([1, 2, 3, 4, 5]).filter_records(lambda x: x > 2)
print(p)            # Pipeline 'ETL_Job' cu 3 înregistrări
print(len(p))       # 3
print(repr(p))      # DataPipeline(name='ETL_Job', records=3)

Pipeline 'ETL_Job' cu 3 înregistrări
3
DataPipeline(name='ETL_Job', records=3)
